# 🎙️ Voxenium Voice Fine-Tuning (Google Colab)

Welcome to the **Voxenium Voice Fine-Tuning Studio**! This notebook runs on Google Colab's free **16GB NVIDIA GPU**.

### 🌟 What this notebook will do:
1. **Set up the GPU Environment** (Checks that we have a 16GB GPU).
2. **Clone Voxenium** (Downloads your updated codebase).
3. **Record or Upload Your Voice** (Upload a few short `.wav` files and transcripts).
4. **Align the Words** (Matches each spoken word to its audio timestamp).
5. **Train the Voice Model** (Adjusts the model's vocal cords to sound like you).
6. **Download Your Checkpoint** (Saves the `.pt` file to your laptop so you can run it on your CPU!).

---
👉 **First Step:** Make sure your Colab is using a GPU: Click **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
# Cell 1: Check GPU and Clone Voxenium
# This verifies we have a free 16GB NVIDIA GPU and downloads our code.

!nvidia-smi

# Clone your repository
!git clone https://github.com/abyyxhek/VOXENIUM.git pocket-tts
%cd pocket-tts

# Install dependencies needed for training
!pip install --quiet soundfile scipy sentencepiece tokenizers transformers tqdm pydantic typer safetensors

### 📁 Step 2: Upload Your Audio Clips

Record 3 to 10 short audio clips of yourself talking (5 to 10 seconds each).
Save them as `01.wav`, `02.wav`, `03.wav`.

Run the cell below to upload them into Colab:

In [ ]:
# Cell 2: Upload Voice Recordings
import os
from google.colab import files

os.makedirs('data/my_voice/wavs', exist_ok=True)
print("Click 'Choose Files' below to upload your .wav recordings:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.wav'):
        os.rename(filename, os.path.join('data/my_voice/wavs', filename))
        print(f"Saved: data/my_voice/wavs/{filename}")

### ✍️ Step 3: Match Text Transcripts with Audio

Type what you said in each clip below. The computer needs to read what your voice is saying.

In [ ]:
# Cell 3: Create Metadata Manifest
# Edit the lines below to match your actual recordings!
import json
import soundfile as sf

voice_samples = [
    {"file": "01.wav", "text": "Hello, my name is Alex and I am building my own voice assistant."},
    {"file": "02.wav", "text": "The quick brown fox jumps over the lazy dog near the riverbank."},
    {"file": "03.wav", "text": "Today is a great day to learn how artificial intelligence models work."}
]

entries = []
for sample in voice_samples:
    path = f"data/my_voice/wavs/{sample['file']}"
    if os.path.exists(path):
        info = sf.info(path)
        entries.append({
            "path": os.path.abspath(path),
            "duration": info.duration,
            "transcript": sample["text"]
        })

# Save training manifest
os.makedirs("data", exist_ok=True)
with open("data/train.jsonl", "w", encoding="utf-8") as f:
    for entry in entries:
        f.write(json.dumps(entry) + "\n")

print(f"Prepared {len(entries)} audio clips for alignment!")

### ⏱️ Step 4: Run CTC Word Alignment

This runs the automated aligner on the GPU to calculate timestamps for each spoken word.

In [ ]:
# Cell 4: Align words to audio timestamps using GPU
!python -m training.scripts.align_data data/train.jsonl data/train_aligned.jsonl --device cuda
print("Alignment complete! Check data/train_aligned.jsonl")

### 🚀 Step 5: Fine-Tune the Voice Model!

Now we launch the training process. The model will adjust its acoustic parameters on the 16GB GPU.

In [ ]:
# Cell 5: Launch Training on 16GB GPU
# You can stop this cell anytime after 500-1000 steps once the voice sounds good!
!python training/train.py training/configs/finetune.yaml

### 💾 Step 6: Download Your Checkpoint to Your Laptop

Once training reaches a checkpoint, run this cell to download your custom voice model (`.pt`) directly to your computer!

In [ ]:
# Cell 6: Download the trained checkpoint
import glob
from google.colab import files

checkpoints = glob.glob("runs/finetune/checkpoint_*.pt")
if checkpoints:
    latest_ckpt = sorted(checkpoints)[-1]
    print(f"Downloading latest checkpoint: {latest_ckpt}")
    files.download(latest_ckpt)
else:
    print("No checkpoint found yet. Wait for training to save a checkpoint!")